# シグモイドネットワークのライブ学習 / Live sigmoid-network training
入力2 → 隠れ層 → 出力1。活性化関数はシグモイドです。

**Colabで「ランタイム → セッションを再起動してすべて実行」→ Start / Resume**。
Startはボタンの処理内で学習を直接実行します。学習中は描画間隔ごとにネットワーク・境界・損失を更新します。**Step +50** は50更新だけ実行します。

実行中に中断する場合はColabの実行停止（■）または「ランタイム → 実行を中断」を使用してください。独立したStopボタンはありません。再開はStart。日本語・英語併記、CPU動作。

In [ ]:
# ライブラリ / Libraries
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass


## 計算 / Computation
$H=\sigma(XW_1+b_1),\quad z=HW_2+b_2,\quad p=\sigma(z)$

二値交差エントロピー / Binary cross-entropy:
$L=\mathrm{mean}[\log(1+e^z)-tz]$.

出力側は $\partial L/\partial z=(p-t)/N$。交差エントロピーとの組合せでは出力シグモイドの微分が整理されます。隠れ層には $H(1-H)$ が明示的に現れます。前のスライドの二乗損失とは損失の選択が異なります。

The output derivative simplifies with cross-entropy; hidden gradients still multiply the sigmoid slope. Full-batch gradient descent is used.

更新方法 / Optimizer: Adam (default) or full-batch gradient descent. Adam rescales gradients using first and second moments; backpropagation itself is unchanged.

In [ ]:
# データ生成：全タスク2次元 / Every task has two input dimensions
def dataset(kind, seed=4, count=240, noise=0.06):
    rng=np.random.default_rng(seed)
    if kind=='XOR (4 points)':
        return np.array([[-1,-1],[1,-1],[-1,1],[1,1.]],float), np.array([[0],[1],[1],[0.]])
    if kind=='XOR clouds':
        y=rng.integers(0,2,count); a=rng.choice([-1,1],count)
        X=np.column_stack([a, a*(1-2*y)])*0.7+rng.normal(0,noise+0.10,(count,2))
    elif kind=='Circles':
        y=np.arange(count)%2; a=rng.uniform(0,2*np.pi,count); r=np.where(y==0,0.4,1.0)
        X=np.column_stack([r*np.cos(a),r*np.sin(a)])+rng.normal(0,noise,(count,2))
    elif kind=='Moons':
        y=np.arange(count)%2; a=rng.uniform(0,np.pi,count)
        X=np.column_stack([np.where(y==0,np.cos(a),1-np.cos(a)),np.where(y==0,np.sin(a),0.5-np.sin(a))])
        X=(X-np.array([0.5,0.25]))/1.3+rng.normal(0,noise,(count,2))
    else:
        y=np.arange(count)%2; r=rng.uniform(0.1,1,count); a=2*np.pi*r+y*np.pi
        X=np.column_stack([r*np.cos(a),r*np.sin(a)])+rng.normal(0,noise*0.5,(count,2))
    return X,y.reshape(-1,1).astype(float)

def sigmoid(z):
    # 安定なシグモイド / Numerically stable sigmoid
    return np.exp(-np.logaddexp(0,-z))

def initialize(width, seed):
    rng=np.random.default_rng(seed)
    # 対称性を崩すランダム初期値 / Random weights break symmetry
    return [rng.normal(0,1,(2,width)),np.zeros((1,width)),rng.normal(0,1/np.sqrt(width),(width,1)),np.zeros((1,1))]

def forward(X,P):
    W,b,V,c=P; H=sigmoid(X@W+b); z=H@V+c
    return H,z,sigmoid(z)

def loss_grad(X,t,P):
    W,b,V,c=P; H,z,p=forward(X,P)
    loss=np.mean(np.logaddexp(0,z)-t*z)
    # 連鎖律：出力 → 隠れ層 / Chain rule: output -> hidden layer
    dz=(p-t)/len(X)
    dH=(dz@V.T)*H*(1-H)
    return loss,[X.T@dH,dH.sum(axis=0,keepdims=True),H.T@dz,dz.sum(axis=0,keepdims=True)]

def train(X,t,width=12,seed=4,lr=1.0,epochs=6000,stride=50):
    P=initialize(width,seed); snapshots=[]; history=[]
    for step in range(epochs+1):
        loss,G=loss_grad(X,t,P); history.append(loss)
        if step%stride==0 or step==epochs:
            snapshots.append((step,[a.copy() for a in P]))
        if step<epochs:
            P=[a-lr*g for a,g in zip(P,G)]
    return snapshots,np.array(history)

# 1更新を独立させ、ライブ表示と検証で共有 / One update shared by live UI and tests
def optimizer_step(X,t,P,opt,lr,moments,step):
    loss,G=loss_grad(X,t,P)
    if opt=='Adam':
        m,v=moments
        for j,g in enumerate(G):
            m[j]=.9*m[j]+.1*g; v[j]=.999*v[j]+.001*g*g
            P[j]-=lr*(m[j]/(1-.9**step))/(np.sqrt(v[j]/(1-.999**step))+1e-8)
    else:
        for a,g in zip(P,G): a-=lr*g
    return P

def zero_moments(P):
    return ([np.zeros_like(a) for a in P],[np.zeros_like(a) for a in P])


## 操作 / Controls
- **Reset**：設定したタスク・初期値で初期化。設定変更後はReset。
- **Start / Resume**：学習を直接実行。完了後はResetで新しい実験。
- **Step +50**：50回更新して停止。まずこれで動作を確認できます。
- 中断：Colabの実行停止（■）またはランタイムの実行中断。学習中に他のウィジェット操作はできません。
- **History**：停止中に学習履歴を表示。再開は最新の重みから。
- **Sample / Neuron**：入力例と、可視化する隠れニューロンを変更。
- 接続線は赤が正、青が負。太さは重みの絶対値（上限あり）。ノードの色と数値は活性値、bはバイアス。

Network, boundary and loss use the same parameters. Scores are training scores. Adam is the default; gradient descent remains available. Errors are printed in the visible output panel.

In [ ]:
# ライブ学習と可視化 / Live training and visualization
import time
import traceback
from matplotlib.patches import Circle
kind=widgets.Dropdown(options=['XOR (4 points)','XOR clouds','Circles','Moons','Spiral'],description='Task')
width=widgets.IntSlider(value=12,min=2,max=24,step=2,description='Hidden')
optimizer=widgets.Dropdown(options=['Adam','Gradient descent'],description='Optimizer')
rate=widgets.FloatText(value=.03,description='Rate')
epochs=widgets.IntSlider(value=3000,min=500,max=15000,step=500,description='Epochs')
seed=widgets.IntSlider(value=4,min=0,max=20,description='Seed')
noise=widgets.FloatSlider(value=.06,min=0,max=.2,step=.02,description='Noise')
interval=widgets.IntSlider(value=50,min=10,max=250,step=10,description='Draw every')
reset=widgets.Button(description='Reset',button_style='info')
start=widgets.Button(description='Start / Resume',button_style='success')
step_button=widgets.Button(description='Step +50',button_style='info')
history=widgets.IntSlider(value=0,min=0,max=0,description='History',continuous_update=False)
sample=widgets.IntSlider(value=0,min=0,max=3,description='Sample',continuous_update=False)
unit=widgets.IntSlider(value=0,min=0,max=11,description='Neuron',continuous_update=False)
status=widgets.HTML(); out=widgets.Output(); state={}
settings=[kind,width,optimizer,rate,epochs,seed,noise,interval]

def snapshot():
    P=state['P']; X,t=state['X'],state['t']; loss=loss_grad(X,t,P)[0]
    acc=float(np.mean((forward(X,P)[2]>=.5)==t))
    state['records'].append((state['step'],[a.copy() for a in P],float(loss),acc))
    history.max=len(state['records'])-1
    history.value=history.max

def draw_network(ax,P,X,index):
    W,b,V,c=P; H,z,p=forward(X[index:index+1],P); count=W.shape[1]
    ypos=np.linspace(.94,.06,count); ins=[.68,.32]
    ax.set(xlim=(-.25,2.65),ylim=(-.04,1.08)); ax.axis('off')
    def edge(a,b,w):
        ax.plot([a[0],b[0]],[a[1],b[1]],color='#d45b41' if w>=0 else '#3276ae',lw=.25+min(abs(float(w)),4)*.45,alpha=.6,zorder=0)
    for j,y in enumerate(ypos):
        for i,iy in enumerate(ins): edge((0,iy),(1,y),W[i,j])
        edge((1,y),(2,.5),V[j,0])
    def node(x,y,label,val,bias=None,input_node=False):
        color='#eeeeee' if input_node else plt.cm.YlOrRd(float(val))
        ax.add_patch(Circle((x,y),.027 if count>16 and x==1 else .035,facecolor=color,edgecolor='#132d52',lw=1.2))
        ax.text(x-.06,y,label,ha='right',va='center',fontsize=8)
        text=f'{float(val):.2f}'
        if bias is not None: text+=f'  b={float(bias):+.2f}'
        ax.text(x+.05,y,text,ha='left',va='center',fontsize=7)
    for i,iy in enumerate(ins): node(0,iy,f'x{i+1}',X[index,i],input_node=True)
    for j,y in enumerate(ypos): node(1,y,f'h{j}',H[0,j],b[0,j])
    node(2,.5,'p',p[0,0],c[0,0])
    ax.set_title(f'2 → {count} → 1 | sample {index}, target {int(state["t"][index,0])}')
    ax.text(0,1.03,'Input',fontsize=10); ax.text(1,1.03,'Sigmoid',fontsize=10); ax.text(2,1.03,'Sigmoid',fontsize=10)
    ax.text(0,-.025,'Edges: red + / blue − | thickness: |weight|',fontsize=9)

def render(change=None):
    if not state or not state['records']: return
    rec=state['records'][min(history.value,len(state['records'])-1)]
    step,P,loss,acc=rec; X,t=state['X'],state['t']; grid=state['grid']; gx,gy=state['mesh']
    H,z,p=forward(grid,P); idx=min(sample.value,len(X)-1)
    with out:
        clear_output(wait=True)
        fig=plt.figure(figsize=(15,9)); gs=fig.add_gridspec(2,2,height_ratios=[1,1.15])
        a=fig.add_subplot(gs[0,0]); im=a.contourf(gx,gy,p.reshape(gx.shape),levels=np.linspace(0,1,21),cmap='RdBu',vmin=0,vmax=1)
        if p.min()<.5<p.max(): a.contour(gx,gy,p.reshape(gx.shape),levels=[.5],colors='black')
        a.scatter(X[:,0],X[:,1],c=t.ravel(),cmap='RdBu',vmin=0,vmax=1,edgecolors='white',s=24)
        a.scatter(*X[idx],s=140,facecolors='none',edgecolors='gold',linewidths=2)
        a.set(title=f'{state["kind"]}: epoch {step}, accuracy {acc:.1%}',xlabel='x1',ylabel='x2'); a.set_aspect('equal'); fig.colorbar(im,ax=a,label='P(class 1)')
        a=fig.add_subplot(gs[0,1]); records=np.array([[r[0],r[2],r[3]] for r in state['records']]); upto=history.value+1
        a.plot(records[:upto,0],records[:upto,1],color='#e87520',label='Loss'); a.set(xlabel='Epoch',ylabel='Binary cross-entropy',title=f'Loss {loss:.5f}'); a.grid(alpha=.2)
        a2=a.twinx(); a2.plot(records[:upto,0],records[:upto,2],color='#168a8a',label='Accuracy'); a2.set(ylim=(0,1.05),ylabel='Training accuracy')
        a=fig.add_subplot(gs[1,0]); draw_network(a,P,X,idx)
        a=fig.add_subplot(gs[1,1]); k=min(unit.value,H.shape[1]-1)
        a.contourf(gx,gy,H[:,k].reshape(gx.shape),levels=np.linspace(0,1,21),cmap='YlOrRd',vmin=0,vmax=1)
        a.scatter(X[:,0],X[:,1],c=t.ravel(),cmap='RdBu',vmin=0,vmax=1,edgecolors='white',s=22)
        a.set(title=f'Hidden neuron {k}: activation 0–1',xlabel='x1',ylabel='x2'); a.set_aspect('equal')
        fig.tight_layout(); display(fig); plt.close(fig)
    status.value=f'<b>{state["opt"]}</b> | epoch {step}/{state["limit"]} | loss {loss:.5f} | training accuracy {acc:.1%} | '+('Running' if state['running'] else 'Paused / Ready')

def lock(running):
    for w in settings+[reset]: w.disabled=running
    start.disabled=running; step_button.disabled=running; history.disabled=running

def initialize_ui(_=None):
    if state.get('running'): return
    if not np.isfinite(rate.value) or rate.value<=0:
        status.value='Rate must be a finite positive number.'; return
    state.clear(); X,t=dataset(kind.value,seed.value,noise=noise.value); P=initialize(width.value,seed.value)
    axis=np.linspace(float(X.min()-.25),float(X.max()+.25),85); gx,gy=np.meshgrid(axis,axis)
    state.update(X=X,t=t,P=P,moments=zero_moments(P),step=0,records=[],running=False,kind=kind.value,opt=optimizer.value,lr=rate.value,limit=epochs.value,every=interval.value,mesh=(gx,gy),grid=np.c_[gx.ravel(),gy.ravel()])
    sample.max=len(X)-1; unit.max=width.value-1; history.value=0; history.max=0
    snapshot(); render()

def run_training(max_updates=None):
    # コールバック内で直接実行。イベントループへの予約に依存しない / Run directly, without scheduling a background coroutine
    if state.get('running'): return
    if state['step']>=state['limit']:
        status.value='Finished. Press Reset for a new run.'; return
    state['running']=True; lock(True)
    end=state['limit'] if max_updates is None else min(state['limit'],state['step']+max_updates)
    error=None
    try:
        status.value='Training started… / 学習開始'
        while state['step']<end:
            for _ in range(min(state['every'],end-state['step'])):
                state['step']+=1
                optimizer_step(state['X'],state['t'],state['P'],state['opt'],state['lr'],state['moments'],state['step'])
            if not all(np.isfinite(a).all() for a in state['P']):
                raise FloatingPointError('Nonfinite weights: lower Rate and Reset.')
            snapshot(); render()
            time.sleep(.02)
    except KeyboardInterrupt:
        error='Interrupted. Start resumes from the latest parameters.'
    except Exception:
        error=traceback.format_exc()
    finally:
        state['running']=False; lock(False); render()
        if error:
            status.value='<b>Stopped / 停止</b>'
            with out: print(error)

def start_training(_=None): run_training()
def step_training(_=None): run_training(50)

def replay(change):
    if not state.get('running'): render()
reset.on_click(initialize_ui); start.on_click(start_training); step_button.on_click(step_training)
history.observe(replay,names='value'); sample.observe(render,names='value'); unit.observe(render,names='value')
def optimizer_preset(change): rate.value=.03 if optimizer.value=='Adam' else 1.0
optimizer.observe(optimizer_preset,names='value')
display(widgets.VBox([widgets.HBox([kind,width]),widgets.HBox([optimizer,rate,epochs]),widgets.HBox([seed,noise,interval]),widgets.HBox([reset,start,step_button]),status,widgets.HBox([history,sample,unit]),out]))
initialize_ui()


## 観察のヒント / What to observe
- **XOR**：隠れニューロンの応答を組み合わせ、1本の直線では分けられない分類を実現します。
- **Circles / Moons**：複数の滑らかな応答を組み合わせて曲がった境界を作ります。
- **Spiral**：少数のニューロンでは難しくなります。幅・学習時間・初期値の影響を比較してください。
- 出力が0または1に近い隠れニューロンでは、傾き $h(1-h)$ が小さくなります。
- 損失が下がっても判定精度が変わらない期間があります。確率の改善と閾値判定は別です。

The hidden layer learns features; the output combines them. Width and initialization affect optimization. Training examples are not a test set. No GPU or external dataset is required.

初期状態は未学習です。Startを押して進行させてください。幅が2など小さい設定では表現力や初期値により学習が進みにくい場合があります。図のネットワークは全ての接続を描き、入力例の切替では重みを変えずに活性値のみを再計算します。

### ボタンが反応しない環境用 / Fallback without widgets
下のセルの `RUN_WITHOUT_BUTTON = True` に変更してセルを実行すると、ボタンを経由せず学習を進められます。 / Set True and run the cell to train without a button.

In [ ]:
# ボタンを使わない代替実行 / Optional direct execution
RUN_WITHOUT_BUTTON = False
if RUN_WITHOUT_BUTTON:
    run_training()